# Homework 5: Supervised Fine-Tuning

In this homework you will **fine-tune a small instruction-tuned language model to answer elementary-school science questions while keeping its general knowledge.** Your model will be evaluated on a hidden test set of multiple-choice science and general-knowledge questions. The hidden science questions were written by course staff and are not taken from any public dataset, so, as in most real deployments, the questions you are graded on will not look exactly like the data you train on. You will upload your trained weights to Hugging Face, and staff will run them on the hidden test set with exactly the code in `hw5_utils.py`.

The model-training part of this homework is open-ended: train a model and upload it to Hugging Face. The paper reading, training code, and training write-up are described in the homework handout. To help you get started, we are providing you with two demo notebooks. This notebook is a standalone **supervised fine-tuning (SFT)** demo. `hw5_rl.ipynb` is a separate standalone demo for reinforcement learning. To train your final model, you can use either notebook as a starting point, or write your own notebook. We also provide `hw5_eval.ipynb` to help you test your submission by running the grader's pipeline on an uploaded model and a sample eval set.

You will fine-tune **Qwen/Qwen2.5-0.5B-Instruct**, a 0.5-billion-parameter model that is small enough for **full fine-tuning** (updating every weight) on a free Colab T4 or Databricks A10. You may also choose to use LoRA fine-tuning, but you will need to merge the weights before uploading (i.e., do not just upload the trained adapter).

Even though the model is small, the pipeline follows the **ML lifecycle** from lecture:

<p align="center">
  <img src="https://az1326.github.io/images/cs189-ml-pipeline.png" width="600">
</p>

## Learning Problem (L)

**Goal:** Decide what behavior we want the model to learn, and from what data.

We want the model to answer multiple-choice science questions. SFT does this by imitation: given a question and its options, the model is trained to produce the correct letter.

Concretely, we will:
- **Load questions** as rows with columns `id, question, A, B, C, D, E, answer`. This is the format the grader uses.
- **Format each row as a chat.** The user message is the question and the options. The assistant message is the correct letter, written `\boxed{A}`.
- **Train on OpenBookQA** elementary-school science questions, converted into that same format, and check progress on `hw5_sample_eval.csv`.

## Model Design (M)

**Goal:** Decide which model we use and how we adapt it.

We use **Qwen/Qwen2.5-0.5B-Instruct**.
- **Architecture:** A transformer causal language model. The architecture stays fixed; the grader rejects anything else.
- **Adaptation:** Full fine-tuning, updating every weight. LoRA is allowed if you merge the adapter back into the model before uploading.

## Optimization (O)

**Goal:** Train the model by minimizing a loss.

We use TRL's `SFTTrainer`:
- **Loss:** Token-level cross-entropy on the assistant's reply, so the model learns to answer rather than to repeat the question.
- **Optimizer:** AdamW. The weights stay in float32, and the matrix multiplications run in bfloat16 or float16 depending on the GPU.
- **Hyperparameters:** Learning rate, batch size, and number of steps, all set in one cell below.

## Predict & Evaluate (P)

**Goal:** Check whether the fine-tuned model behaves as desired.

After training, we:
- **Run inference** with the grader's prompt, tokenizer, and greedy decoding.
- **Score accuracy** by parsing `\boxed{X}` from the generation.
- **Compare** accuracy before and after fine-tuning, then upload the weights for grading.

### Rules

**What you CAN change:**
- **Training and validation data:** mix in other datasets and/or build your own question CSVs for training and/or validation.
- **Training method:** SFT, RL, or both; full fine-tuning or LoRA (merged before upload); any hyperparameters.

**What you CANNOT change:**
- **The model:** you must train `Qwen/Qwen2.5-0.5B-Instruct`. Uploads that are not this model (different size or architecture, resized embeddings, unmerged LoRA, quantized weights) are rejected automatically before grading.
- **How your model is evaluated:** `hw5_utils.py` fixes the prompt (`build_mcq_prompt`), the base model's tokenizer and chat template, greedy decoding with 128 new tokens, float32 weights, and the answer parser (`parse_choice_from_boxed`). The prompt asks for one sentence and then `\boxed{X}`, which fits easily in 128 tokens. Only the first `\boxed{X}` counts, and anything past 128 tokens is cut off, so the boxed answer has to land inside the budget. Do not edit `hw5_utils.py`: the grader uses its own clean copy so any changes you make will not make it to the grader.
  - You can make `EVAL_BATCH_SIZE` in `hw5_utils.py` smaller if you run into out-of-memory errors during evaluation.

**Submission:** a private Hugging Face repository URL where you uploaded the model, the commit SHA of the upload, and a fine-grained read-only Hugging Face token to access the model, all submitted to a [Google Form](https://forms.gle/HAoRwfytwQEdQHMg8). We will update your model's performance on the hidden test set nightly. See the end of this notebook for details.

## Part 0: Environment Setup

Select a GPU runtime (Colab: **Runtime > Change runtime type > T4 GPU**; Databricks: an A10 with base environment **AI v6**). If you have a local accelerator you want to use, create a Python 3.12 environment and `pip install -r requirements.txt`, but note that staff cannot help much with technical support for local development.

In [ ]:
import os
import sys

IS_COLAB = "google.colab" in sys.modules
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IS_COLAB:
    print("Colab environment. Make sure the HW_DIR below matches what you used in the startup notebook.")
    print("If you are asked to restart the runtime, you do not need to do so.")
    print("If you decide to restart the runtime, you will need to re-run this cell.")
    from google.colab import drive
    drive.mount("/content/drive")
    HW_DIR = "/content/drive/MyDrive/CS189/hw/hw5"
    os.chdir(HW_DIR)
    %pip install -q -r colab_requirements.txt
elif IS_DATABRICKS:
    print("Databricks environment. Make sure your base environment is AI v6.")
    %pip install -q -r databricks_requirements.txt
else:
    print("Local development. Make sure your kernel has everything in requirements.txt.")

In [ ]:
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer

import hw5_utils

device = hw5_utils.pick_device()
print("Using device:", device)  # should be cuda (or mps on a Mac)

### Model Precision

The released `Qwen/Qwen2.5-0.5B-Instruct` weights are stored in bfloat16. We will load them in a more precise **float32** format for training and let the trainer run the math in an efficient 16-bit format. This is known as *mixed precision* training.

- **float32 weights**: Fine-tuning updates are tiny compared to the weights themselves. A bfloat16 weight keeps only about 3 significant digits, so a learning-rate-sized update added to a 16-bit weight can round to no change. Keeping weights in float32 means every update is actually captured.
- **16-bit math**: Running the forward and backward pass in 16-bit provides significant speed and memory savings.

The hardware available determines which 16-bit format the math uses. Newer hardware support **bfloat16**, which covers the exact same numerical range as float32. Older GPUs only support **float16**, which cannot represent values very close to zero. To prevent tiny numbers in the middle of the backward pass from rounding to zero, float16 requires loss scaling (multiplying the loss to scale up the math during backpropagation, then dividing it back down at the end).

The utility function `hw5_utils.training_precision()` automatically inspects your machine and selects the best setup:

| Hardware | Compute dtype | Why |
|----------|---------------|-----|
| A10, L4, A100, H100, RTX 30-series or newer (NVIDIA Ampere+) | bfloat16 | Newer GPUs support bfloat16 |
| T4, RTX 20-series, other pre-Ampere NVIDIA GPUs | float16 | Older GPUs lack bfloat16 support |
| Apple Silicon M2 and later | bfloat16 | M2+ chips support bfloat16 |
| Apple Silicon M1 | float16 | M1 chips lack bfloat16 support |
| CPU | float32 | CPUs offer no useful speedup from 16-bit math |

You will upload your final model in float32, and the grader evaluates submissions in float32; your precision setting will not affect the grading process. However, training will likely be more stable if your machine supports bfloat16.

In [ ]:
PRECISION, PRECISION_NOTE = hw5_utils.training_precision()
print("Training with", PRECISION_NOTE)

---
## Part 1: Configuration

In [ ]:
# --- Model (YOU CANNOT CHANGE THIS) ---
MODEL_NAME = hw5_utils.BASE_MODEL
MODEL_REVISION = hw5_utils.BASE_REVISION  # pinned so the tokenizer and chat template match the grader

# --- Data ---
EVAL_CSV_PATH = "hw5_sample_eval.csv"  # sample science questions in the hidden test format
NUM_TRAIN_QUESTIONS = 112  # OpenBookQA training questions used by this short demo

# --- Training (feel free to adjust!) ---
TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
WARMUP_STEPS = 5
MAX_STEPS = 30
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
LR_SCHEDULER_TYPE = "linear"
OPTIM = "adamw_torch"
SEED = 189

OUTPUT_DIR = "sft_model"  # merged checkpoint uploaded at the end

set_seed(SEED)

---
## Part 2: Data

### The question format

Every question the grader sees is one row of a CSV with these columns (`hw5_utils.MCQ_COLUMNS`):

| Column | Content |
|--------|---------|
| `id` | unique string |
| `question` | question text |
| `A` ... `E` | the five options |
| `answer` | the correct letter, A-E |

### Evaluation data

`hw5_sample_eval.csv` is a sample in exactly this format, and `hw5_utils.load_mcq_csv` is the loader the grader uses on the hidden test. Any validation set you build should use the same format, so you can evaluate it here and in `hw5_eval.ipynb`.

In [ ]:
eval_questions = hw5_utils.load_mcq_csv(EVAL_CSV_PATH)
print(f"Loaded {len(eval_questions)} evaluation questions from {EVAL_CSV_PATH}")
pd.DataFrame(eval_questions).head()

### What the model sees

For each question, the grader builds a user message with `build_mcq_prompt` and wraps it in the base model's chat template. The template adds Qwen's default system message and the special tokens `<|im_start|>` and `<|im_end|>` that mark who is speaking. The model then generates the assistant's reply, and `parse_choice_from_boxed` extracts the letter.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
eval_prompt = [{"role": "user", "content": hw5_utils.build_mcq_prompt(eval_questions[0])}]
print(tokenizer.apply_chat_template(eval_prompt, tokenize=False, add_generation_prompt=True))

### Training data (OpenBookQA)

We train on a small random sample from the training split of **OpenBookQA**, a dataset of elementary-school science questions. Each OpenBookQA question tests one fact from a small "open book" of about 1,300 science facts; the dataset's `additional` config lists the fact behind each question. OpenBookQA questions have four lettered options, so `hw5_utils.openbookqa_to_mcq` converts each one to the five-option format above, using "none of the above" as option E. You should convert every dataset you use to this format as that is the format of the hidden test set the grader will use.

Each training example is a **prompt-completion pair** in chat format: the user message from `build_mcq_prompt`, and an assistant reply you choose. `SFTTrainer` applies the chat template to both, and by default computes the loss only on the completion, so the model only learns to answer, not how to ask questions.

`make_completion` below decides what the model is trained to say. In this demo it is just `\boxed{X}`, but you could modify it as long as **you have `\boxed{X}`** (as that is how the grader extracts the answer) and **you keep it within 128 tokens** (as that is the generation budget). For example, you can write out reasoning before the box (a form of chain-of-thought) or synthesize completions with another model (distillation). 

In [ ]:
obqa = load_dataset("allenai/openbookqa", "main", split="train")
obqa = obqa.shuffle(seed=SEED).select(range(NUM_TRAIN_QUESTIONS))
train_questions = [hw5_utils.openbookqa_to_mcq(row, i) for i, row in enumerate(obqa)]
print(f"Converted {len(train_questions)} OpenBookQA questions")
display(pd.DataFrame(train_questions).head(3))

def make_completion(q):
    """The assistant's training target for one question. Edit this to try other targets, e.g. CoT."""
    # The prompt asks for one sentence before the box; this demo trains directly on the answer.
    return f"\\boxed{{{q['answer']}}}"

def to_sft_example(q):
    prompt = [{"role": "user", "content": hw5_utils.build_mcq_prompt(q)}]
    completion = [{"role": "assistant", "content": make_completion(q)}]
    return {"prompt": prompt, "completion": completion}

train_dataset = Dataset.from_list([to_sft_example(q) for q in train_questions])
train_dataset[0]

You can save any question list in the grader's format with `hw5_utils.write_mcq_csv(questions, "my_questions.csv")` and load it back with `hw5_utils.load_mcq_csv`.

---
## Part 3: Load the Model and Measure a Baseline

`hw5_utils.evaluate_model` scores a model using the same settings the grader will use. We run it here on the base model (before fine-tuning) for a baseline.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, dtype=torch.float32)
model.to(device)

eval_setup = hw5_utils.prepare_eval(eval_questions)
baseline = hw5_utils.evaluate_model(model, eval_questions, eval_setup)
print(f"Baseline accuracy: {baseline['accuracy']:.3f} ({baseline['num_correct']}/{baseline['num_questions']})")
pd.DataFrame(baseline["predictions"]).head()

---
## Part 4: Supervised Fine-Tuning

`SFTTrainer` minimizes the token-level cross-entropy of the completion given the prompt.
- `learning_rate`, `per_device_train_batch_size`, `gradient_accumulation_steps`, `max_steps`: the usual optimization hyperparameters. The effective batch size is `TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS`.
- `bf16` / `fp16`: the mixed-precision setting chosen in Part 0.

The values for these hyperparameters were defined above in the Configuration section.

With float16, the first few logged steps often show `grad_norm: inf` and `learning_rate: 0`. That is loss scaling calibrating itself: the trainer multiplies the loss by a large factor so small float16 gradients do not underflow, skips any step whose gradients overflow, and lowers the factor until they fit. Training then proceeds normally.

In [ ]:
mps_kwargs = {}
if device.type == "mps":
    # Pad to fixed lengths to avoid an MPS memory leak: MPS caches a kernel per batch shape.
    # https://huggingface.co/docs/transformers/main/perf_train_special#limit-graph-cache-growth
    mps_kwargs["pad_to_multiple_of"] = 128

sft_config = SFTConfig(
    output_dir="sft_checkpoints",
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    optim=OPTIM,
    logging_steps=1,
    save_strategy="no",
    seed=SEED,
    report_to="none",
    **PRECISION,
    **mps_kwargs,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
trainer.train()

---
## Part 5: Evaluate After SFT

In [ ]:
finetuned = hw5_utils.evaluate_model(model, eval_questions, eval_setup)
print(f"Baseline accuracy:   {baseline['accuracy']:.3f}")
print(f"Fine-tuned accuracy: {finetuned['accuracy']:.3f} ({finetuned['num_correct']}/{finetuned['num_questions']})")

details = pd.DataFrame(eval_questions)[["id", "question", "answer"]].merge(
    pd.DataFrame(finetuned["predictions"]), on="id"
)
details.head(10)

---
## Possible Next Steps

This demo is a simple tutorial, not a strong submission. A few places to start looking:

- **Option E is never correct in the training data.** `openbookqa_to_mcq` turns each four-option OpenBookQA question into five options by adding "none of the above" as E, so every training answer is A-D. A model trained only on this data never sees any problems where E is the correct answer.
- **The training target has no reasoning.** `make_completion` trains the model to reply with the box alone. A common training paradigm is to learn to generate some reasoning before committing to an answer, and indeed the eval prompt does try to elicit some reasoning.

---
## Part 6: Save and Upload Your Model

Your submission must be **one merged checkpoint**: `model.safetensors` with exactly the same tensor names and shapes as `Qwen/Qwen2.5-0.5B-Instruct`, in float32, bfloat16, or float16. If you trained with LoRA, `hw5_utils.save_checkpoint` merges the adapter (`merge_and_unload()`) before saving. Do not resize the token embeddings, quantize the model, or change the architecture.

### Step 1: Save the merged checkpoint

In [ ]:
hw5_utils.save_checkpoint(model, tokenizer, OUTPUT_DIR)

### Step 2: Upload to a private Hugging Face repository

1. Create a free account at [huggingface.co](https://huggingface.co).
2. Set `HF_REPO_ID` to `<your-username>/cs189-hw5` and run the cell. It creates the repository as **private**, uploads `OUTPUT_DIR`, and prints the commit SHA.

Every upload creates a new commit. Staff grade the exact commit SHA you submit, so later uploads do not change your submission unless you resubmit with the new SHA.

If you need the commit SHA from an earlier submission, or lost the commit SHA, see the [Hugging Face Guide](https://docs.google.com/document/d/1rawrv4npw9sNGej0K6R36ZwHBN7PzIEqQqs99ymQNl0/edit?usp=sharing) for instructions on obtaining the SHA from the Hugging Face website.

*Note:* If the `login()` call asks for a token, create a token with **Write** access at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and paste it when asked.

In [ ]:
from huggingface_hub import login

HF_REPO_ID = "your-username/cs189-hw5"  # TODO: replace with your Hugging Face username

login()
COMMIT_SHA = hw5_utils.upload_checkpoint(OUTPUT_DIR, HF_REPO_ID)

### Step 3: Create the read-only token you submit

Staff need to read your private repository (and nothing else) to download and evaluate your model. At [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens), click **Create new token** and choose:
- Token type: **Fine-grained**
- Under **Repositories permissions**, select only your `HF_REPO_ID` repository and check **Read access to contents of selected repos**
- Leave every other permission unchecked
- See the [Hugging Face Guide](https://docs.google.com/document/d/1rawrv4npw9sNGej0K6R36ZwHBN7PzIEqQqs99ymQNl0/edit?usp=sharing) for a detailed walkthrough.

### Step 4: Check your upload with the grader's pipeline

Open `hw5_eval.ipynb`, enter the repository URL, the commit SHA printed above, and the read-only token, and run it. It performs every step the grader performs, so a submission that scores there will be graded the same way (but on a hidden test set).

### Step 5: Submit to Google Form

Submit the repository URL, the commit SHA, and the fine-grained read-only token to the [HW5 Model Submission Form](https://forms.gle/HAoRwfytwQEdQHMg8).
- Keep the repository private.
- Do not delete or revoke the submitted token until grades are released and regrade requests are resolved, or staff will not be able to download your model.
- **Never submit a token with write permissions.** This is especially relevant if you needed to create a write token in Step 2.

You are allowed **one submission per day**. If you submit multiple times to the Google form in the same day, we will only accept your **last submission of the day**.

### Step 6: Check Pensive for results
Every night at midnight, staff will download the **latest submission of the day** from every student and evaluate the model on the hidden test set. Once every submitted model has been evaluated (hopefully the next morning, but may take longer), we update scores. **To view the score of your most recently processed model, upload your training code to the HW5 Leaderboard assignment on Pensive.**

After the deadline, your score on this part of HW5 will be based on your **best recorded model performance**. Partial credit is available, and we will release exact scoring cutoffs at a later date.